In [ ]:
import torch # Importing required modules
import torch.nn as nn 
import numpy as np
import re
from collections import Counter

In [ ]:
# Read file # reading data from local directory
with open("shakespeare.txt", 'r') as f: 
    text = f.read()


print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


In [ ]:
text = text.lower() # converting to lowercase 
text = re.sub(r'[^a-z\s]', '', text)

words = text.split()

print("Total words:", len(words))

Total words: 202619


In [ ]:
words = words[:80000] # limiting to 80k words for faster training
print("Reduced words:", len(words))

Reduced words: 80000


In [ ]:
vocab = sorted(set(words)) 
 
word_to_idx = {w: i for i, w in enumerate(vocab)}
idx_to_word = {i: w for w, i in word_to_idx.items()}

vocab_size = len(vocab)
print("Vocab size:", vocab_size)

Vocab size: 7565


In [ ]:
seq_length = 10   

X = []
y = []

for i in range(len(words) - seq_length):
    seq = words[i:i+seq_length]
    target = words[i+seq_length]

    X.append([word_to_idx[w] for w in seq])
    y.append(word_to_idx[target])

X = torch.tensor(X) # Converting to PyTorch tensors
y = torch.tensor(y)

print("Shape:", X.shape)

Shape: torch.Size([79990, 10])


In [ ]:
perm = torch.randperm(X.size(0)) # Shuffling data
X = X[perm]
y = y[perm]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

X = X.to(device) # Moving tensors to device
y = y.to(device)

Using device: cuda


In [ ]:
class LSTMModel(nn.Module): # Defining the LSTM architecture
    def __init__(self, vocab_size):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, 200)

        self.lstm = nn.LSTM(
            input_size=200,
            hidden_size=512,
            num_layers=2,
            batch_first=True,
            dropout=0.3
        )

        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, vocab_size)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]

        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.fc2(out)

        return out


model = LSTMModel(vocab_size).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss() # Defining loss function
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) # Defining optimizer

In [ ]:
epochs = 70 # trained upto 70 epochs
batch_size = 128

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]
        yb = y[i:i+batch_size]

        optimizer.zero_grad()

        output = model(xb)
        loss = criterion(output, yb)

        loss.backward()

        # 🔥 prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5)

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}") # initial 35 and now 70 epochs 

Epoch 1/70, Loss: 4224.3281
Epoch 2/70, Loss: 3931.6072
Epoch 3/70, Loss: 3748.4522
Epoch 4/70, Loss: 3576.9307
Epoch 5/70, Loss: 3405.8560
Epoch 6/70, Loss: 3239.6002
Epoch 7/70, Loss: 3094.5531
Epoch 8/70, Loss: 2956.4652
Epoch 9/70, Loss: 2844.2438
Epoch 10/70, Loss: 2717.3533
Epoch 11/70, Loss: 2600.0475
Epoch 12/70, Loss: 2485.4916
Epoch 13/70, Loss: 2376.0425
Epoch 14/70, Loss: 2271.6464
Epoch 15/70, Loss: 2170.5180
Epoch 16/70, Loss: 2084.7651
Epoch 17/70, Loss: 2000.8252
Epoch 18/70, Loss: 1924.5563
Epoch 19/70, Loss: 1848.8138
Epoch 20/70, Loss: 1787.7079
Epoch 21/70, Loss: 1728.0118
Epoch 22/70, Loss: 1665.5362
Epoch 23/70, Loss: 1618.5637
Epoch 24/70, Loss: 1576.5647
Epoch 25/70, Loss: 1530.6274
Epoch 26/70, Loss: 1502.7024
Epoch 27/70, Loss: 1473.6450
Epoch 28/70, Loss: 1440.2424
Epoch 29/70, Loss: 1404.9764
Epoch 30/70, Loss: 1385.5774
Epoch 31/70, Loss: 1367.6290
Epoch 32/70, Loss: 1336.4964
Epoch 33/70, Loss: 1314.6910
Epoch 34/70, Loss: 1280.5052
Epoch 35/70, Loss: 1256

In [ ]:
torch.save({ # Saving the model and related info
    'model_state_dict': model.state_dict(),
    'word_to_idx': word_to_idx,
    'idx_to_word': idx_to_word,
    'seq_length': seq_length
}, "shakespeare_lstm.pt")

print("Model saved!")

Model saved!


In [ ]:
def predict_next(text_seq, temperature=0.8): # Function to predict next word given a text sequence
    model.eval()
    
    words_input = text_seq.lower().split()[-seq_length:]
    seq = [word_to_idx.get(w, 0) for w in words_input]

    if len(seq) < seq_length:
        seq = [0]*(seq_length - len(seq)) + seq

    seq = torch.tensor(seq).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(seq)
        probs = torch.softmax(output / temperature, dim=1).cpu().numpy().flatten()

    pred = np.random.choice(len(probs), p=probs)

    return idx_to_word[pred]

In [ ]:
def generate_text(seed, next_words=10): # Function to generate text given a seed and number of next words to predict
    result = seed

    for _ in range(next_words):
        next_word = predict_next(result)
        result += " " + next_word

    return result

In [15]:
print(generate_text("to be or not to", 15))

to be or not to wail the will of their infirmity sir keeps a dearer cushion made a theft more
